In [1]:
import os
import glob
import cv2
import numpy as np
import tensorflow as tf
from sklearn.metrics.pairwise import polynomial_kernel
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Conv2D,
    Conv2DTranspose,
    Dense,
    Reshape,
    Embedding,
    BatchNormalization,
)

# --- Configuration ---
DATA_DIR = "/kaggle/input/datasets/imrankhan77/nct-crc-he-100k/NCT-CRC-HE-100K"
MODEL_SAVE_DIR = "/kaggle/working/"
TARGET_CLASS = "STR"

LATENT_VECTOR_DIM = 128
IMG_SIZE = 128
NUM_SAMPLES = 250
BATCH_SIZE = 50
EPOCHS_TO_TEST = [60, 70, 80, 90, 100]

# =============================================================================
# 1. Rebuild the Generator Architecture
# =============================================================================
class ConditionalGenerator(Model):
    """cGAN generator: (z, label) -> 128x128x3."""
    def __init__(
        self,
        n_classes,
        latent_dim,
        embed_dim=128,
        base_channels=512,
        img_channels=3,
        name="ConditionalGenerator",
        **kwargs,
    ):
        super().__init__(name=name, **kwargs)
        self.n_classes = n_classes
        self.latent_dim = latent_dim
        self.embed_dim = embed_dim
        self.base_channels = base_channels

        self.label_embedding = Embedding(n_classes, embed_dim, name="g_label_embedding")
        self.fc = Dense(4 * 4 * base_channels, name="g_fc")
        self.reshape = Reshape((4, 4, base_channels), name="g_reshape")

        self.g_tconv_8 = Conv2DTranspose(512, 4, strides=2, padding="same", use_bias=False, name="g_tconv_8")
        self.g_bn_8 = BatchNormalization(name="g_bn_8")

        self.g_tconv_16 = Conv2DTranspose(256, 4, strides=2, padding="same", use_bias=False, name="g_tconv_16")
        self.g_bn_16 = BatchNormalization(name="g_bn_16")

        self.g_tconv_32 = Conv2DTranspose(128, 4, strides=2, padding="same", use_bias=False, name="g_tconv_32")
        self.g_bn_32 = BatchNormalization(name="g_bn_32")

        self.g_tconv_64 = Conv2DTranspose(64, 4, strides=2, padding="same", use_bias=False, name="g_tconv_64")
        self.g_bn_64 = BatchNormalization(name="g_bn_64")
        
        self.g_tconv_128 = Conv2DTranspose(32, 4, strides=2, padding="same", use_bias=False, name="g_tconv_128")
        self.g_bn_128 = BatchNormalization(name="g_bn_128")

        self.g_out = Conv2D(
            img_channels,
            7,
            padding="same",
            activation="tanh",
            name="g_out_conv",
        )

    def call(self, inputs, training=False):
        z, label = inputs
        le = self.label_embedding(label)
        le = tf.squeeze(le, axis=1)
        x = tf.concat([z, le], axis=-1)
        x = self.fc(x)
        x = self.reshape(x)

        x = tf.nn.relu(self.g_bn_8(self.g_tconv_8(x), training=training))
        x = tf.nn.relu(self.g_bn_16(self.g_tconv_16(x), training=training))
        x = tf.nn.relu(self.g_bn_32(self.g_tconv_32(x), training=training))
        x = tf.nn.relu(self.g_bn_64(self.g_tconv_64(x), training=training))
        x = tf.nn.relu(self.g_bn_128(self.g_tconv_128(x), training=training))

        return self.g_out(x)

# =============================================================================
# 2. Evaluation Logic
# =============================================================================
def fast_tiff_load_for_inception(path_bytes):
    """Loads a TIFF image using OpenCV and resizes it for InceptionV3."""
    path = path_bytes.decode("utf-8")
    img = cv2.imread(path)
    if img is None:
        return np.zeros((299, 299, 3), dtype=np.float32)
        
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (299, 299), interpolation=cv2.INTER_AREA)
    
    # Inception preprocess_input expects inputs in [0, 255] float32
    return img.astype(np.float32)

def evaluate_cgan_epochs_for_class(target_class, epochs_to_test, num_samples=250, batch_size=50):
    
    class_names = sorted(
        d for d in os.listdir(DATA_DIR)
        if os.path.isdir(os.path.join(DATA_DIR, d)) and d.upper() != "BACK"
    )
    if target_class not in class_names:
        raise ValueError(f"Class '{target_class}' not found in {class_names}")
    
    target_label_idx = class_names.index(target_class)
    n_classes = len(class_names)
    print(f"Target Class: {target_class} (Label Index: {target_label_idx})")

    # --- Load Real Images ---
    print(f"\nLoading real images for {target_class}...")
    target_dir = os.path.join(DATA_DIR, target_class)
    all_files = glob.glob(os.path.join(target_dir, "*.*"))
    selected_files = all_files[:num_samples]
    
    def load_and_prep_real(path):
        img = tf.numpy_function(fast_tiff_load_for_inception, [path], tf.float32)
        img.set_shape([299, 299, 3])
        return preprocess_input(img)

    real_dataset = tf.data.Dataset.from_tensor_slices(selected_files)
    real_dataset = real_dataset.map(load_and_prep_real, num_parallel_calls=tf.data.AUTOTUNE)
    real_dataset = real_dataset.batch(batch_size)

    # --- Load InceptionV3 ---
    print("Loading InceptionV3 Feature Extractor...")
    inception = InceptionV3(include_top=False, pooling='avg', input_shape=(299, 299, 3))
    inception.trainable = False
    
    # --- Extract features for REAL images ---
    print("Extracting features for REAL images...")
    real_features = []
    for real_batch in real_dataset:
        real_features.append(inception.predict(real_batch, verbose=0))
        
    real_features = np.concatenate(real_features, axis=0)
    
    d = real_features.shape[1]
    m = real_features.shape[0]
    k_xx = polynomial_kernel(real_features, real_features, degree=3, gamma=1./d, coef0=1.0)
    np.fill_diagonal(k_xx, 0)
    sum_k_xx = np.sum(k_xx)

    # --- Initialize the empty Generator shell ---
    generator = ConditionalGenerator(n_classes=n_classes, latent_dim=LATENT_VECTOR_DIM)
    # Dummy pass to initialize variables before loading weights
    _ = generator([tf.zeros((1, LATENT_VECTOR_DIM)), tf.zeros((1, 1), dtype=tf.int32)])

    kid_results = {}
    num_batches = num_samples // batch_size

    # --- Evaluate each epoch ---
    for epoch in epochs_to_test:
        print(f"\n=========================================")
        print(f" Evaluating Epoch {epoch} ")
        print(f"=========================================")
        
        filepath = os.path.join(MODEL_SAVE_DIR, f"/kaggle/input/models/snowbubble0/gen-{epoch}/keras/default/1/generator_epoch_{epoch}.weights.h5")
        
        if not os.path.exists(filepath):
            print(f"File not found: {filepath}. Skipping...")
            continue
            
        print(f" Loading weights from: {filepath}")
        # Load weights into the initialized shell
        generator.load_weights(filepath)
        
        fake_features = []
        for i in range(num_batches):
            print(f"  Generating and processing fake batch {i+1}/{num_batches}...")
            
            z = tf.random.normal((batch_size, LATENT_VECTOR_DIM))
            labels = tf.fill([batch_size, 1], target_label_idx)
            
            fake_images = generator([z, labels], training=False)
            
            fake_images = (fake_images + 1.0) / 2.0 * 255.0
            fake_images_resized = tf.image.resize(fake_images, (299, 299))
            fake_images_prepped = preprocess_input(fake_images_resized)
            
            fake_features.append(inception.predict(fake_images_prepped, verbose=0))

        fake_features = np.concatenate(fake_features, axis=0)
        
        k_yy = polynomial_kernel(fake_features, fake_features, degree=3, gamma=1./d, coef0=1.0)
        k_xy = polynomial_kernel(real_features, fake_features, degree=3, gamma=1./d, coef0=1.0)
        
        np.fill_diagonal(k_yy, 0)
        kid = (sum_k_xx + np.sum(k_yy)) / (m * (m - 1)) - 2 * np.mean(k_xy)
        
        kid_scaled = kid 
        
        print(f" Epoch {epoch} KID Score: {kid_scaled:.5f}")
        kid_results[epoch] = kid_scaled

    print(f"\n FINAL KID SCORE SUMMARY ({target_class})")
    print("-" * 35)
    for ep, score in kid_results.items():
        print(f"Epoch {ep}: \t {score:.5f}")
    print("-" * 35)
        
    return kid_results

# Execute the evaluation
all_kid_scores = evaluate_cgan_epochs_for_class(
    target_class=TARGET_CLASS, 
    epochs_to_test=EPOCHS_TO_TEST, 
    num_samples=NUM_SAMPLES, 
    batch_size=BATCH_SIZE
)

2026-06-02 23:11:21.460761: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780441881.651092      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780441881.706299      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780441882.170329      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780441882.170393      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780441882.170398      58 computation_placer.cc:177] computation placer alr

Target Class: STR (Label Index: 6)

Loading real images for STR...


I0000 00:00:1780441895.985089      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Loading InceptionV3 Feature Extractor...
87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
Extracting features for REAL images...


I0000 00:00:1780441906.031815     123 service.cc:152] XLA service 0x7835b8002630 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780441906.031871     123 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1780441907.291565     123 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1780441914.501458     123 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.



 Evaluating Epoch 60 
 Loading weights from: /kaggle/input/models/snowbubble0/gen-60/keras/default/1/generator_epoch_60.weights.h5
  Generating and processing fake batch 1/5...
  Generating and processing fake batch 2/5...
  Generating and processing fake batch 3/5...
  Generating and processing fake batch 4/5...
  Generating and processing fake batch 5/5...
 Epoch 60 KID Score: 0.12041

 Evaluating Epoch 70 
 Loading weights from: /kaggle/input/models/snowbubble0/gen-70/keras/default/1/generator_epoch_70.weights.h5
  Generating and processing fake batch 1/5...
  Generating and processing fake batch 2/5...
  Generating and processing fake batch 3/5...
  Generating and processing fake batch 4/5...
  Generating and processing fake batch 5/5...
 Epoch 70 KID Score: 0.10932

 Evaluating Epoch 80 
 Loading weights from: /kaggle/input/models/snowbubble0/gen-80/keras/default/1/generator_epoch_80.weights.h5
  Generating and processing fake batch 1/5...
  Generating and processing fake batch 2